In [ ]:
import os, numpy as np, pandas as pd
from pathlib import Path
from pandas.tseries.offsets import MonthEnd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

from math import log

import xgboost as xgb

# ========= CONFIG =========
INTERIM_PATH = Path("../data/interim")
DATA_VERSION = "v2"  # change to "v3" to run the ablation with pageviews

# Label columns
Y_REG = "excess_log_next"   # regression target: next-month excess log return
Y_CLS = "y_class_next"      # classification label: 1 if next-month excess > 0

# Required baseline columns
ID_COLS   = ["date", "ticker"]
BASE_COLS = ["l_stock","l_bench","ex_log","close","volume"]
STRUCT    = ["country_UK"]  # sector dummies already in v2/v3 if you kept them
MACRO_LAGS = ["vix_lvl","dxy_lvl","us10y_lvl","dex_usuk","brent_d3m"]  # lagged in v4; if not, we’ll shift below


In [ ]:
# Pick the file
if DATA_VERSION == "v3":
    path = INTERIM_PATH / "features_stocks_v3.parquet"
else:
    path = INTERIM_PATH / "features_stocks_v2.parquet"
df = pd.read_parquet(path)
df["date"] = pd.to_datetime(df["date"]) + MonthEnd(0)
df = df.sort_values(["date","ticker"]).reset_index(drop=True)

# Build labels if not already present
if Y_REG not in df.columns or Y_CLS not in df.columns:
    print("Creating prediction labels...")
    # Next-month stock & bench log returns
    g = df.groupby("ticker")
    l_stock_next = g["l_stock"].shift(-1)
    l_bench_next = g["l_bench"].shift(-1)
    df[Y_REG] = (l_stock_next - l_bench_next)  # next-month EXCESS log return
    df[Y_CLS] = (df[Y_REG] > 0).astype("int8")
    print("Labels created successfully")

# Remove final month per ticker
df = df.dropna(subset=[Y_REG]).copy()

Creating prediction labels...
Labels created successfully


In [3]:
df["date"] = pd.to_datetime(df["date"]) + MonthEnd(0)

REQUIRED_MACRO = ["vix_lvl","dxy_lvl","us10y_lvl","brent_d3m","dex_usuk"]

# If any missing, merge from macro_monthly.parquet
missing = [c for c in REQUIRED_MACRO if c not in df.columns]
if missing:
    macro_p = INTERIM_PATH / "macro_monthly.parquet"
    macro = pd.read_parquet(macro_p)
    macro["date"] = pd.to_datetime(macro["date"]) + MonthEnd(0)
    df = df.merge(macro, on="date", how="left", validate="many_to_one")
    print("Merged missing macros:", missing)

# Alias map (promote alternates to canonical)
ALIAS_MAP = {
    "vix_lvl":   ["vix","VIX","VIXCLS","vix_level"],
    "dxy_lvl":   ["dxy","DTWEXBGS","broad_usd"],
    "us10y_lvl": ["us10y","DGS10","us10y_yield"],
    "brent_d3m": ["brent_d3m","brent_3m","brent_chg3m","brent_delta3m"],
    "dex_usuk":  ["DEXUSUK","gbpusd","GBPUSD"],
}
for canon, alts in ALIAS_MAP.items():
    if canon not in df.columns:
        for a in alts:
            if a in df.columns:
                df[canon] = df[a]
                print(f"Aliased {a} → {canon}")
                break

# Coalesce any _x/_y variants to a single canonical column
def coalesce_to(df, base):
    cols = [c for c in df.columns if c == base or c.startswith(base + "_")]
    if not cols:
        return df
    df[base] = pd.concat(
        [df[c] for c in cols if c != base] + ([df[base]] if base in cols else []),
        axis=1
    ).bfill(axis=1).iloc[:,0]
    for c in cols:
        if c != base:
            df.drop(columns=c, inplace=True, errors="ignore")
    return df

for base in REQUIRED_MACRO:
    df = coalesce_to(df, base)

print("After coalesce →")
for c in REQUIRED_MACRO:
    print(f"{c:10s}", c in df.columns, "| non-null%:", df[c].notna().mean() if c in df.columns else "—")

After coalesce →
vix_lvl    True | non-null%: 1.0
dxy_lvl    True | non-null%: 0.951417004048583
us10y_lvl  True | non-null%: 1.0
brent_d3m  True | non-null%: 1.0
dex_usuk   True | non-null%: 1.0


In [4]:
# Sliding window through dates
years = df["date"].dt.year
FOLDS = [
    {"train_end": 2016, "val": 2017},
    {"train_end": 2017, "val": 2018},
    {"train_end": 2018, "val": 2019},
    {"train_end": 2019, "val": 2020},
    {"train_end": 2020, "val": 2021},
    {"train_end": 2021, "val": 2022},
]
TEST_YEARS = [2023, 2024]


In [ ]:
# Regime flags
def roll_quantile(series, win, q):
    return series.rolling(win, min_periods=win).quantile(q)

def z12(series):
    mu = series.rolling(12, min_periods=12).mean()
    sd = series.rolling(12, min_periods=12).std()
    return (series - mu) / sd

def add_regime_flags_per_fold(dtr, dva, win=60):
    # TRAIN thresholds (past-only)
    vix_p75_tr     = roll_quantile(dtr["vix_lvl"], win, 0.75)
    abs_br_p80_tr  = roll_quantile(dtr["brent_d3m"].abs(), win, 0.80)

    # Merge row-wise thresholds by date (last value per date inside train)
    th = (pd.DataFrame({
            "date": dtr["date"],
            "vix_p75": vix_p75_tr,
            "abs_br_p80": abs_br_p80_tr,
        })
        .groupby("date", as_index=False).tail(1))
    dtr = dtr.merge(th, on="date", how="left")
    dva = dva.merge(th, on="date", how="left")

    # Build flags for train & val, then shift(1) to predict t+1
    for d in (dtr, dva):
        # If macro levels in your file weren’t lagged in 04, they are still t-values;
        # we build flags from t-values and then SHIFT FLAGS so the model uses info up to t-1.
        d["high_vix"]      = (d["vix_lvl"] >= d["vix_p75"]).astype("int8")
        d["oil_shock"]     = (d["brent_d3m"].abs() >= d["abs_br_p80"]).astype("int8")
        d["strong_dollar"] = (z12(d["dxy_lvl"])   >  1.0).astype("int8")
        d["strong_gbp"]    = (z12(d["dex_usuk"])  >  1.0).astype("int8")

        # Bear tape: 12m benchmark momentum using t-1 info
        d["bear_tape"] = (
            d.groupby("ticker")["l_bench"]
             .shift(1).rolling(12, min_periods=12).sum() < 0
        ).astype("int8")

        # Vol regime: top quartile of 3m realised vol of benchmark (trailing 5y)
        ben_vol3 = d["l_bench"].shift(1).rolling(3, min_periods=3).std()
        vol_p75  = roll_quantile(ben_vol3, win, 0.75)
        d["vol_regime"] = (ben_vol3 >= vol_p75).astype("int8")

        # Country interaction
        if "country_UK" in d.columns:
            d["strong_gbp_uk"] = (d["strong_gbp"] * d["country_UK"]).astype("int8")
        else:
            d["strong_gbp_uk"] = 0

        # Strict LAG of all flags to avoid using same-month info
        lag_flags = ["high_vix","oil_shock","strong_dollar","strong_gbp","bear_tape","vol_regime","strong_gbp_uk"]
        for c in lag_flags:
            d[c] = d[c].shift(1)

    # Drop rows with NaNs created by lag at the head
    dtr = dtr.dropna(subset=["high_vix","oil_shock","strong_dollar","strong_gbp","bear_tape","vol_regime","strong_gbp_uk"])
    dva = dva.dropna(subset=["high_vix","oil_shock","strong_dollar","strong_gbp","bear_tape","vol_regime","strong_gbp_uk"])
    return dtr, dva


In [ ]:
# Recency Weight
def recency_weights(dates, half_life_months=24):
    # Coerce to datetime and handle NaT/empty safely
    d = pd.to_datetime(dates, errors="coerce")
    n = len(d)
    if n == 0:
        return np.array([], dtype=float)
    if d.notna().sum() == 0:
        return np.ones(n, dtype=float)  # neutral weights if all NaT

    # Start month = earliest valid month in the slice
    m0 = d[d.notna()].min().to_period("M")
    # Compute age in months; fill NaT ages with median age (or 0) to keep array finite
    age = (d.dt.to_period("M") - m0).astype("Int64")
    if age.notna().any():
        fill_val = int(age[age.notna()].median())
    else:
        fill_val = 0
    age = age.fillna(fill_val).astype(int)

    w = np.exp(-log(2) * (age / half_life_months))
    return w / w.mean()



In [7]:
# Build feature set
def build_feature_list(df):
    # start from everything except ids & labels
    exclude = set(ID_COLS + [Y_REG, Y_CLS])
    cols = [c for c in df.columns if c not in exclude]

    # You can keep it lean to start; include regime flags and your core signals
    core_price = [
        "mom_log_3","mom_log_6","mom_log_12","rev_1m","vol_6","vol_12",
        "sharpe_12","near_high_12","dist_sma_12","drawdown_12",
        "beta_36","tracking_error_12","up_cap_36","down_cap_36","up_down_cap_ratio_36",
        "liquidity"
    ]
    structure = ["country_UK","sec_Energy","sec_Financials","sec_Healthcare","sec_Industrials","sec_Staples","sec_Tech"]
    macro_lvls = ["vix_lvl","dxy_lvl","us10y_lvl","dex_usuk","brent_d3m"]
    regime = ["high_vix","oil_shock","strong_dollar","strong_gbp","bear_tape","vol_regime","strong_gbp_uk"]
    trends = [c for c in ["gt_ma3","gt_z12","gt_spike","pv_ma3","pv_z12","pv_spike"] if c in df.columns]

    feat = [c for c in core_price + structure + macro_lvls + regime + trends if c in df.columns]
    # Drop any accidental duplicates
    feat = list(dict.fromkeys(feat))
    return feat


# QA

In [8]:
print("n cols:", len(df.columns))
print(sorted(df.columns)[:60])   # first 60 names
print("...")

REQUIRED_MACRO = ["vix_lvl","dxy_lvl","us10y_lvl","brent_d3m","dex_usuk"]
for c in REQUIRED_MACRO:
    print(f"{c:12s} present? {c in df.columns}  non-null%:",
          df[c].notna().mean() if c in df.columns else "—")


n cols: 14
['brent_d3m', 'close', 'date', 'dex_usuk', 'dxy_lvl', 'ex_log', 'excess_log_next', 'l_bench', 'l_stock', 'ticker', 'us10y_lvl', 'vix_lvl', 'volume', 'y_class_next']
...
vix_lvl      present? True  non-null%: 1.0
dxy_lvl      present? True  non-null%: 0.951417004048583
us10y_lvl    present? True  non-null%: 1.0
brent_d3m    present? True  non-null%: 1.0
dex_usuk     present? True  non-null%: 1.0


In [ ]:
# Model Run
def eval_rank_ic(df_sub, y_reg_col, y_pred):
    # monthly Spearman between y and pred, then mean
    m = df_sub.assign(pred=y_pred).groupby(df_sub["date"].dt.to_period("M"))
    ics = []
    for _, g in m:
        if g[y_reg_col].nunique() > 1 and g["pred"].nunique() > 1:
            ics.append(g[[y_reg_col,"pred"]].corr(method="spearman").iloc[0,1])
    return np.nanmean(ics) if ics else np.nan

def decile_spread(df_sub, y_reg_col, y_score):
    # long top decile, short bottom decile on next-month EXCESS returns (y_reg)
    d = df_sub.copy()
    d["score"] = y_score
    # within each month, pick deciles
    rets = []
    for dt, g in d.groupby(d["date"].dt.to_period("M")):
        if len(g) < 10:
            continue
        q = np.quantile(g["score"], [0.1, 0.9])
        long = g[g["score"] >= q[1]][y_reg_col].mean()
        short = g[g["score"] <= q[0]][y_reg_col].mean()
        rets.append(long - short)
    if not rets:
        return np.nan, np.nan, 0
    arr = np.asarray(rets)
    t = arr.mean() / (arr.std(ddof=1) / np.sqrt(len(arr))) if arr.std(ddof=1) > 0 else np.nan
    return arr.mean(), t, len(arr)

def run_fold(train_end, val_year):
    tr = df[years <= train_end].copy()
    va = df[years == val_year].copy()
    # add fold-aware regime flags
    tr, va = add_regime_flags_per_fold(tr, va, win=60)
    # feature list (includes flags)
    FEATS = build_feature_list(tr)

    # ---- Build matrices ----
    Xtr = tr[FEATS].dropna()
    ytr = tr.loc[Xtr.index, Y_CLS]
    wtr = recency_weights(tr.loc[Xtr.index, "date"], half_life_months=24)

    Xva = va[FEATS].dropna()
    yva = va.loc[Xva.index, Y_CLS]
    yva_reg = va.loc[Xva.index, Y_REG]

    # skip fold if train/val is empty (after lag/warmups) ----
    if Xtr.empty or Xva.empty:
        print(f"[skip] train_end={train_end}, val_year={val_year} "
              f"(n_train={len(Xtr)}, n_val={len(Xva)})")
        return ({
            "train_end": train_end, "val_year": val_year,
            "logit_auc": np.nan, "xgb_auc": np.nan,
            "logit_rankIC": np.nan, "xgb_rankIC": np.nan,
            "logit_spread": np.nan, "xgb_spread": np.nan,
            "months": 0, "n_train": len(Xtr), "n_val": len(Xva),
            "n_feats": len(FEATS)
        }, FEATS)


    # ---------- Logistic (baseline) ----------
    logit = LogisticRegression(max_iter=2000, C=1.0, solver="lbfgs")
    logit.fit(Xtr, ytr, sample_weight=wtr)
    p_val = logit.predict_proba(Xva)[:,1]
    auc_l = roc_auc_score(yva, p_val)

    # threshold tuning on validation (balanced accuracy)
    grid = np.linspace(0.2, 0.8, 121)
    best_t, best_bal = 0.5, -1
    for t in grid:
        bal = balanced_accuracy_score(yva, (p_val >= t).astype(int))
        if bal > best_bal:
            best_bal, best_t = bal, t

    ic_l   = eval_rank_ic(va.loc[Xva.index], Y_REG, p_val)
    sprd_l, tstat_l, n_m = decile_spread(va.loc[Xva.index], Y_REG, p_val)

    # ---------- XGBoost ----------
    xgbc = xgb.XGBClassifier(
        max_depth=3, n_estimators=700, learning_rate=0.07,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_lambda=1.0, tree_method="hist", n_jobs=-1, random_state=42
    )
    xgbc.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xva, yva)], verbose=False)
    px_val = xgbc.predict_proba(Xva)[:,1]
    auc_x  = roc_auc_score(yva, px_val)

    best_tx, best_bax = 0.5, -1
    for t in grid:
        bal = balanced_accuracy_score(yva, (px_val >= t).astype(int))
        if bal > best_bax:
            best_bax, best_tx = bal, t

    ic_x   = eval_rank_ic(va.loc[Xva.index], Y_REG, px_val)
    sprd_x, tstat_x, _ = decile_spread(va.loc[Xva.index], Y_REG, px_val)

    out = {
        "train_end": train_end, "val_year": val_year,
        "logit_auc": auc_l, "logit_best_t": best_t, "logit_balacc": best_bal,
        "logit_rankIC": ic_l, "logit_spread": sprd_l, "logit_t": tstat_l, "months": n_m,
        "xgb_auc": auc_x, "xgb_best_t": best_tx, "xgb_balacc": best_bax,
        "xgb_rankIC": ic_x, "xgb_spread": sprd_x, "xgb_t": tstat_x,
        "n_train": len(Xtr), "n_val": len(Xva), "n_feats": len(FEATS)
    }
    return out, FEATS



# -------- Run the folds -----------
results = []
last_feats = None
for f in FOLDS:
    r, feats = run_fold(f["train_end"], f["val"])
    results.append(r)
    last_feats = feats

res = pd.DataFrame(results)
print(res)
print("\nMean (validation across folds)")
print(res[["logit_auc","xgb_auc","logit_rankIC","xgb_rankIC","logit_spread","xgb_spread"]].mean())


   train_end  val_year  logit_auc  logit_best_t  logit_balacc  logit_rankIC  \
0       2016      2017   0.492938         0.440      0.524153      0.050557   
1       2017      2018   0.580079         0.455      0.590266     -0.038433   
2       2018      2019   0.452564         0.505      0.532479     -0.071891   
3       2019      2020   0.595157         0.465      0.590171     -0.084275   
4       2020      2021   0.510882         0.485      0.543245      0.030000   
5       2021      2022   0.461825         0.400      0.518226      0.082169   

   logit_spread   logit_t  months   xgb_auc  xgb_best_t  xgb_balacc  \
0      0.018353  1.405722      11  0.579661       0.440    0.571751   
1      0.000385  0.027448      11  0.456989       0.490    0.527589   
2     -0.011354 -1.009681      11  0.519801       0.535    0.565100   
3      0.003450  0.261739      11  0.638462       0.275    0.604131   
4      0.019611  0.674001      11  0.468485       0.755    0.530667   
5     -0.014565 -0.8

In [10]:
# Test the fixed version with a single fold
print("=== Testing Fixed Version with Single Fold ===")

# First, let's check what functions are available
print("Available functions:")
print([name for name in globals() if 'run_fold' in name])

# Test with just one fold first
test_fold = FOLDS[0]  # 2016 train, 2017 val
print(f"Testing fold: train_end={test_fold['train_end']}, val_year={test_fold['val']}")

# Check if we have the data loaded
print(f"Data loaded: {'df' in globals()}")
if 'df' in globals():
    print(f"Data shape: {df.shape}")
    print(f"Date range: {df['date'].min()} to {df['date'].max()}")

# Try to run the original function first
try:
    r, feats = run_fold(test_fold["train_end"], test_fold["val"])
    print("✅ SUCCESS! Original function worked")
    print(f"Results: {r}")
    print(f"Features used: {len(feats)}")
except NameError as e:
    print(f"❌ FUNCTION NOT DEFINED: {e}")
    print("Please run the earlier cells first to define the functions")
except Exception as e:
    print(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()


=== Testing Fixed Version with Single Fold ===
Available functions:
['run_fold']
Testing fold: train_end=2016, val_year=2017
Data loaded: True
Data shape: (2470, 14)
Date range: 2005-02-28 00:00:00 to 2025-08-31 00:00:00
✅ SUCCESS! Original function worked
Results: {'train_end': 2016, 'val_year': 2017, 'logit_auc': 0.4929378531073446, 'logit_best_t': np.float64(0.44000000000000006), 'logit_balacc': 0.5241525423728813, 'logit_rankIC': np.float64(0.050556781301963935), 'logit_spread': np.float64(0.01835294815940927), 'logit_t': np.float64(1.405722358225836), 'months': 11, 'xgb_auc': 0.5796610169491525, 'xgb_best_t': np.float64(0.44000000000000006), 'xgb_balacc': 0.5717514124293785, 'xgb_rankIC': np.float64(0.05649327265389756), 'xgb_spread': np.float64(0.007672868622774229), 'xgb_t': np.float64(0.5187117030503992), 'n_train': 1310, 'n_val': 119, 'n_feats': 12}
Features used: 12
